### Installation

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

fourbit_models = [
    "unsloth/Qwen3-1.7B-unsloth-bnb-4bit", # Qwen 14B 2x faster
    "unsloth/Qwen3-4B-unsloth-bnb-4bit",
    "unsloth/Qwen3-8B-unsloth-bnb-4bit",
    "unsloth/Qwen3-14B-unsloth-bnb-4bit",
    "unsloth/Qwen3-32B-unsloth-bnb-4bit",

    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/Phi-4",
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/orpheus-3b-0.1-ft-unsloth-bnb-4bit" # [NEW] We support TTS models!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length = 2048,   # Context length - can be longer, but uses more memory
    load_in_4bit = True,     # 4bit uses much less memory
    load_in_8bit = False,    # A bit more accurate, uses 2x memory
    full_finetuning = False,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.8: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

unsloth/qwen3-4b-instruct-2507-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,           # Choose any number > 0! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,  # Best to choose alpha = rank or rank*2
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,   # We support rank stabilized LoRA
    loftq_config = None,  # And LoftQ
)

Unsloth 2026.3.8 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


<a name="Data"></a>
### Data Prep


In [ ]:
from datasets import load_dataset

# 1. Load the text file (assuming 1 chunk per line from our cleaning script)
raw_dataset = load_dataset("text", data_files={"train": "cleaned_dataset.txt"})["train"]

# 2. Split into Train (95%) and Eval (5%)
# With 530k lines, 5% is plenty (~26k lines) to verify generalization
dataset = raw_dataset.train_test_split(test_size=0.05, seed=3407)

print(f"Training rows: {len(dataset['train'])}")
print(f"Evaluation rows: {len(dataset['test'])}")

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset['train'],
    eval_dataset = dataset['test'], # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 30,
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use TrackIO/WandB etc
        padding_free  = False, # Set to True if > 17 GB VRAM
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/2457 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/130 [00:00<?, ? examples/s]

Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,457 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 66,060,288 of 4,088,528,384 (1.62% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,2.344900
2,2.496200
3,2.281600
4,2.397400
5,2.391000
6,2.414100
7,2.001200
8,2.229400
9,2.264400
10,2.229500


# Save the weights

In [ ]:
# GGUF (for CPU fast inference)
model.save_pretrained_gguf("gguf_model", tokenizer, quantization_method="q4_k_m")

# HF merged (for flexibility)
model.save_pretrained_merged("qwen_psych_cpu_model_q6", tokenizer, save_method="merged_16bit")# 🔥 much better )

Unsloth: Merging model weights to 16-bit format...


config.json: 0.00B [00:00, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [03:40<00:00, 110.19s/it]


Unsloth: Merge process complete. Saved to `/content/qwen_psych_cpu_model_q6`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q6_k'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['qwen_psych_cpu_model_q6_gguf/qwen3-4b-instruct-2507.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q6_k. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['qwen_psych_cpu_model_q6_gguf/qwen3-4b-instruct-2507.Q6_K.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model qwen_psych_cpu_model_q6_gguf/qwen3-4b-instruct-2507.Q6_K.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to qwen_psych_cpu_model_q6_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f qwen_psych_cpu_model_q6_gguf/Modelfile


{'save_directory': 'qwen_psych_cpu_model_q6',
 'gguf_directory': 'qwen_psych_cpu_model_q6_gguf',
 'gguf_files': ['qwen_psych_cpu_model_q6_gguf/qwen3-4b-instruct-2507.Q6_K.gguf'],
 'modelfile_location': 'qwen_psych_cpu_model_q6_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

# Inference

In [ ]:
!pip install llama-cpp-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 MB 16.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.4 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.16-cp312-cp312-linux_x86_64.whl size=4503266 sha256=f4e858e35294e84d9dd134f0916ce76f0ae14acbd83ee032803666aa49179f27
  Stored in directory: /root/.cache/pip/wheels/90/82/ab/8784ee3fb99ddb07fd36a679ddbe63122cc07718f6c1eb3be8
Successfully built llama-cpp-python


In [ ]:
prompt =f"""
<|system|>
You are a friendly, curious, and fun Educational Psychologist for kids.
- Talk directly to Shekar.
- Use emojis ⚙️⛓️ and "wondering" language.
- Never use formal headings like "Section 1" or "Comparative Questions."
- Compare the gears to real-world things (like a big slow elephant and a fast little mouse).

<|user|>
Here is Shekar's gear activity: {summary}.
Ask him 3 fun, reflective questions to help him learn!"""

In [ ]:
from transformers import AutoTokenizer
from llama_cpp import Llama

# 1. Load the tokenizer from your local folder (from your previous image)
# This is necessary to format the prompt correctly for Qwen
tokenizer = AutoTokenizer.from_pretrained("/content/qwen_psych_cpu_model")

# 2. Load the GGUF model
llm = Llama(
    model_path = "/content/qwen_psych_cpu_model_gguf/qwen3-4b.Q4_K_M.gguf",
    n_ctx = 1024,      # Increased slightly for better reasoning
    n_threads = 2,
    n_batch = 512,
    verbose = False
)

# 3. Use the Chat Template (This prevents "Thinking" leaks)
messages = [
    {"role": "system", "content": "You are a friendly Science Buddy. Give a direct response to Shekar. No internal monologue."},
    {"role": "user", "content": "Shekar built a gear system with 35 and 14 teeth and a chain. Ask him 3 fun questions!"}
]

# This adds the special <|im_start|> and <|im_end|> tokens
full_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# 4. Run the model
response = llm(
    full_prompt,
    max_tokens = 512,
    # CRITICAL: These stop tokens tell the model to STOP after the assistant finishes
    stop = ["<|im_end|>", "<|endoftext|>", "<|im_start|>"],
    temperature = 0.7,
    repeat_penalty = 1.2
)

print(response["choices"][0]["text"].strip())


llama_context: n_ctx_per_seq (1024) < n_ctx_train (40960) -- the full capacity of the model will not be utilized


<think>
Okay, so Shekar built a gear system with 35 and 14 teeth connected by a chain. Let me think about how to ask fun questions here.

First, I remember that when gears are meshed together, the number of teeth determines their speed ratio. The formula is usually (Number of Teeth on Driven Gear) / (Number of Teeth on Driving Gear). But wait, since it's connected by a chain instead of another gear, maybe there's something different here? Chains typically connect sprockets with teeth that engage with the links of the chain, so each tooth engages one link. So if you have two gears connected by a chain, they would be sprockets. But Shekar used 35 and 14 teeth—maybe those are the number of teeth on the sprockets? Or is it about something else?

Wait, maybe he's using regular gears (with interlocking toothed wheels) with chains connected to them. Then the chain would have links that move around the two gears. But in a standard gear system with a chain and another wheel, you'd need both a s